In [1]:
from pathlib import Path
print(Path.cwd())

C:\Users\karol\Bakalarka26\prakticka_cast\notebooks\02_correlations


In [2]:
from datetime import datetime
from pathlib import Path
import platform, sys
import sys
sys.path.append(str(Path.cwd().parents[1]))  # pridá koreň projektu (prakticka_cast) na sys.path


DOMAIN = "corr"             # "ts" | "corr" | "geo"
DATASET_NAME = "wdi"     # napr. "noaa_demo" | "wdi_demo" | "osm_sk_demo"
LIBRARY = "seaborn"        # "matplotlib" | "seaborn" | "plotly" | "geopandas_matplotlib"
TASK_ID = "korelacna_matica"
N_RUNS = 10
SEED = 42
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")


In [3]:
import os, time, json, csv
import numpy as np, pandas as pd, statistics as stats
import matplotlib, matplotlib.pyplot as plt, seaborn as sns
import plotly
import plotly.express as px, plotly.graph_objects as go


try:
    import geopandas as gpd
except Exception:
    gpd = None
try:
    import yaml
except Exception:
    yaml = None

plt.rcParams.update({
    "figure.dpi": 120, "figure.figsize": (8, 4.5),
    "axes.titlesize": 12, "axes.labelsize": 10,
    "legend.fontsize": 9, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "font.family": "DejaVu Sans"
})
sns.set_theme(style="whitegrid")
px.defaults.template = "plotly_white"
px.defaults.width = 800
px.defaults.height = 450

In [4]:
from pathlib import Path
import plotly.io as pio

PROJECT_ROOT = Path.cwd().parents[1]
OUT_ROOT = PROJECT_ROOT / "outputs"

FIG_DIR = OUT_ROOT / "figures"
HTML_DIR = OUT_ROOT / "html"
METRICS_DIR = OUT_ROOT / "metrics"
TABLES_DIR = OUT_ROOT / "tables"

CFG_PLOTS = {"dpi": 120, "svg": True}

for d in [FIG_DIR, HTML_DIR, METRICS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DOMAIN_MAP = {"ts": "time_series", "corr": "correlations", "geo": "geo"}
DOMAIN_DIR = DOMAIN_MAP.get(DOMAIN, DOMAIN)
LIB_DIR = LIBRARY.replace("_", "")

FIG_SUBDIR = FIG_DIR / DOMAIN_DIR / LIB_DIR
HTML_SUBDIR = HTML_DIR / DOMAIN_DIR

FIG_SUBDIR.mkdir(parents=True, exist_ok=True)
HTML_SUBDIR.mkdir(parents=True, exist_ok=True)

# pre Plotly geo export PNG/SVG
PLOTLY_TOPOJSON_DIR = PROJECT_ROOT / "assets" / "plotly_topojson"
if PLOTLY_TOPOJSON_DIR.exists():
    pio.defaults.topojson = str(PLOTLY_TOPOJSON_DIR.resolve())


def file_basename(prefix: str, variant: str, ext: str) -> Path:
    name = f"{DOMAIN}_{LIBRARY}_{DATASET_NAME}_{TASK_ID}_{variant}_{TIMESTAMP}.{ext}"
    return (HTML_SUBDIR if ext.lower() == "html" else FIG_SUBDIR) / name

class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.elapsed = (time.perf_counter() - self.t0) * 1000.0

def file_size_kb(path: Path):
    try:
        return os.path.getsize(path) / 1024.0
    except OSError:
        return None

def write_metrics(row: dict, csv_path: Path = METRICS_DIR / "metrics.csv"):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    exists = csv_path.exists()
    with open(csv_path, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            w.writeheader()
        w.writerow(row)

def count_loc_from_callable(fn) -> int:
    import inspect, textwrap
    try:
        src = textwrap.dedent(inspect.getsource(fn))
        lines = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
        return len(lines)
    except Exception:
        return None

def runtime_meta():
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "matplotlib": matplotlib.__version__,
        "seaborn": sns.__version__,
        "plotly": plotly.__version__
    }

In [5]:
from src.data.wdi import load_wdi_correlations

WDI_YEARS = [2018, 2019, 2020, 2021, 2022]
WDI_INDICATORS = [
    "GDP per capita (current US$)",
    "Life expectancy at birth, total (years)",
    "Urban population (% of total population)",
    "Individuals using the Internet (% of population)",
]

def load_data(domain: str, dataset_name: str):
    if domain != "corr":
        raise ValueError("Tento notebook je určený pre korelačné dáta (DOMAIN='corr').")

    df_raw = load_wdi_correlations(years=WDI_YEARS, indicators=WDI_INDICATORS)
    corr_matrix = df_raw.corr(numeric_only=True)
    corr_matrix.index.name = "Indikátor"
    corr_matrix.columns.name = "Indikátor"
    return corr_matrix

DF = load_data(DOMAIN, DATASET_NAME)
display(DF)

Indikátor,GDP per capita (current US$),Individuals using the Internet (% of population),"Life expectancy at birth, total (years)",Urban population (% of total population)
Indikátor,,,,
GDP per capita (current US$),1.000000,0.575510,0.618026,0.446677
Individuals using the Internet (% of population),0.575510,1.000000,0.851720,0.727028
"Life expectancy at birth, total (years)",0.618026,0.851720,1.000000,0.609962
Urban population (% of total population),0.446677,0.727028,0.609962,1.000000


In [6]:
def plot_baseline(df):
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(df, ax=ax, cmap="Blues", cbar=True)
    ax.set_title("Korelačná matica indikátorov WDI")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    return fig, "mpl"

In [7]:
def plot_final(df):
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        df,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        vmin=-1,
        vmax=1,
        center=0,
        linewidths=.5,
        cbar_kws={"shrink": .8},
        ax=ax
    )
    ax.set_title("Korelačná matica socio-ekonomických indikátorov (WDI 2018–2022)")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    return fig, "mpl"

In [8]:
def render_and_time(make_fig_callable, df, variant: str, n_runs: int = 5) -> dict:
    times, out_files = [], []

    html_path = None
    png_path = None
    svg_path = None

    # warm-up beh, nezapočítava sa do metrík
    fig, kind = make_fig_callable(df)
    if kind == "mpl":
        plt.close(fig)

    for _ in range(n_runs):
        with Timer() as t:
            fig, kind = make_fig_callable(df)
        times.append(t.elapsed)

        if kind == "mpl":
            plt.close(fig)

    fig, kind = make_fig_callable(df)
    export_elapsed_ms = None

    if kind == "mpl":
        png_path = file_basename("plot", variant, "png")
        with Timer() as t:
            fig.savefig(png_path, dpi=CFG_PLOTS.get("dpi", 120), bbox_inches="tight")
        export_elapsed_ms = t.elapsed
        plt.close(fig)
        out_files.append(png_path)

        if CFG_PLOTS.get("svg", True):
            svg_path = png_path.with_suffix(".svg")
            fig, _ = make_fig_callable(df)
            fig.savefig(svg_path, bbox_inches="tight")
            plt.close(fig)
            out_files.append(svg_path)

    elif kind == "plotly":
        html_path = file_basename("plot", variant, "html")
        with Timer() as t:
            fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
        export_elapsed_ms = t.elapsed
        out_files.append(html_path)

        try:
            png_path = file_basename("plot", variant, "png")
            fig.write_image(png_path, scale=2)
            out_files.append(png_path)

            if CFG_PLOTS.get("svg", True):
                svg_path = file_basename("plot", variant, "svg")
                fig.write_image(svg_path)
                out_files.append(svg_path)
        except Exception as e:
            print("Upozornenie: PNG/SVG export (Plotly) preskočený:", e)

    return {
        "build_ms_median": stats.median(times),
        "build_ms_p95": np.percentile(times, 95),
        "export_ms": export_elapsed_ms,
        "saved": str(html_path or png_path) if (html_path or png_path) else None,
        "html_path": str(html_path) if html_path else None,
        "png_path": str(png_path) if png_path else None,
        "svg_path": str(svg_path) if svg_path else None,
        "html_size_kb": round(file_size_kb(html_path), 2) if html_path else None,
        "png_size_kb": round(file_size_kb(png_path), 2) if png_path else None,
        "svg_size_kb": round(file_size_kb(svg_path), 2) if svg_path else None,
        "all_files": [str(p) for p in out_files]
    }

loc_baseline = count_loc_from_callable(plot_baseline)
loc_final    = count_loc_from_callable(plot_final)
res_baseline = render_and_time(plot_baseline, DF, "baseline", n_runs=N_RUNS)
res_final    = render_and_time(plot_final,    DF, "final",    n_runs=N_RUNS)

meta = runtime_meta()
common = {
    "timestamp": TIMESTAMP, "domain": DOMAIN, "dataset": DATASET_NAME, "library": LIBRARY, "task_id": TASK_ID,
    "python": meta["python"], "platform": meta["platform"],
    "matplotlib_ver": meta["matplotlib"], "seaborn_ver": meta["seaborn"], "plotly_ver": meta["plotly"]
}

write_metrics({
    **common,
    "variant": "baseline",
    "build_ms_median": round(res_baseline["build_ms_median"], 2),
    "build_ms_p95": round(res_baseline["build_ms_p95"], 2),
    "export_ms": round(res_baseline["export_ms"] or 0.0, 2),
    "loc": loc_baseline,
    "saved": res_baseline["saved"],
    "html_path": res_baseline["html_path"],
    "png_path": res_baseline["png_path"],
    "svg_path": res_baseline["svg_path"],
    "html_size_kb": res_baseline["html_size_kb"],
    "png_size_kb": res_baseline["png_size_kb"],
    "svg_size_kb": res_baseline["svg_size_kb"]
})

write_metrics({
    **common,
    "variant": "final",
    "build_ms_median": round(res_final["build_ms_median"], 2),
    "build_ms_p95": round(res_final["build_ms_p95"], 2),
    "export_ms": round(res_final["export_ms"] or 0.0, 2),
    "loc": loc_final,
    "saved": res_final["saved"],
    "html_path": res_final["html_path"],
    "png_path": res_final["png_path"],
    "svg_path": res_final["svg_path"],
    "html_size_kb": res_final["html_size_kb"],
    "png_size_kb": res_final["png_size_kb"],
    "svg_size_kb": res_final["svg_size_kb"]
})

print("Hotovo. Uložené:")
print("  baseline:", res_baseline["saved"])
print("  final   :", res_final["saved"])
print("Metriky ->", METRICS_DIR / "metrics.csv")

Hotovo. Uložené:
  baseline: C:\Users\karol\Bakalarka26\prakticka_cast\outputs\figures\correlations\seaborn\corr_seaborn_wdi_korelacna_matica_baseline_2026-04-01_11-05-31.png
  final   : C:\Users\karol\Bakalarka26\prakticka_cast\outputs\figures\correlations\seaborn\corr_seaborn_wdi_korelacna_matica_final_2026-04-01_11-05-31.png
Metriky -> C:\Users\karol\Bakalarka26\prakticka_cast\outputs\metrics\metrics.csv


In [9]:
m = pd.read_csv(METRICS_DIR / "metrics.csv")

subset = m[
    (m["timestamp"] == TIMESTAMP) &
    (m["domain"] == DOMAIN) &
    (m["dataset"] == DATASET_NAME) &
    (m["library"] == LIBRARY) &
    (m["task_id"] == TASK_ID)
].copy()

display(subset[[
    "variant",
    "build_ms_median",
    "build_ms_p95",
    "export_ms",
    "loc",
    "html_size_kb",
    "png_size_kb",
    "svg_size_kb",
    "saved",
    "html_path",
    "png_path",
    "svg_path"
]])

print("Metrics exists AFTER:", (METRICS_DIR / "metrics.csv").exists())
print("Metrics path:", METRICS_DIR / "metrics.csv")

,variant,build_ms_median,build_ms_p95,export_ms,loc,html_size_kb,png_size_kb,svg_size_kb,saved,html_path,png_path,svg_path
4,baseline,57.51,103.64,135.50,7,NaN,94.35,61.61,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,NaN,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,C:\Users\karol\Bakalarka26\prakticka_cast\outp...
5,final,66.40,84.28,144.94,18,NaN,125.35,75.34,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,NaN,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,C:\Users\karol\Bakalarka26\prakticka_cast\outp...


Metrics exists AFTER: True
Metrics path: C:\Users\karol\Bakalarka26\prakticka_cast\outputs\metrics\metrics.csv
